____
# INTERPOLATE AND ROTATE SWOT DATA ON COLOC POINTS

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import dask.dataframe as dd
import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_2km, add_mask_inside_swot, add_grid_metrics, build_swath_polygon
#from diagnosis import drifters_sources

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

In [2]:
if True:
    from dask.distributed import Client
    from dask_jobqueue import PBSCluster
    from dask import config

    config.set({"distributed.comm.timeouts.connect": "200s"})
    cluster = PBSCluster(cores=28, processes=28, walltime="03:00:00")
    # cluster = PBSCluster(cores=20, processes=20, walltime='02:00:00')#8
    w = cluster.scale(jobs=2)
else:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()

client = Client(cluster)
client

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.scheduler.transition-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.comm.recent-messages-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  w

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: http://10.148.0.219:8787/status,
Dashboard: http://10.148.0.219:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.148.0.219:34003,Workers: 0
Dashboard: http://10.148.0.219:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [73]:
cluster.close()

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/

_______________
# CHOOSE coloc sources

In [3]:
drifters_sources = 'all_med_variational_10min_v0.nc'

# Drifters param
dt = '10d'
drifter_preprocess = 'spectral_decomp' # '', 'spectral_decomp', 'low_pass'

if drifter_preprocess == True : drifters_sources = 'spectral_decomp_'+ drifters_sources

colocs_source = f'{dt}_{drifter_preprocess}_{drifters_sources}'.replace('.nc', '.csv')

# Drifters
ddf = dd.read_csv(os.path.join(zarr_dir,'coloc_files', 'drifters', f'drifterscoloc_'+colocs_source), dtype={'drifter_id':str}, parse_dates=['datetime']).set_index('row_number').repartition(npartitions=56)

ddf = ddf[(~((ddf.pass_number==3)& (ddf.cycle_number==568))) & (~((ddf.pass_number==16)&((ddf.cycle_number ==508)|(ddf.cycle_number ==513)|(ddf.cycle_number ==534)|(ddf.cycle_number ==554)|(ddf.cycle_number ==568))))].persist()

df = ddf.compute()

In [4]:
ddf =ddf.persist()

_________
# Functions

In [9]:
ggd_variables = ['cvl_mean_dynamic_topography_cnes_cls_22',
                 'cvl_mean_sea_surface_cnes_22_hybrid',
                 'cvl_ocean_tide_fes_2022',
                 'cvl_ssha_reference',
                 'duacs_ssha_karin_2_calibrated',
                 'duacs_ssha_karin_2_filtered',]


variables =[#'ancillary_surface_classification_flag',
    'cross_track_distance',
    'distance_to_coast',
    'duacs_editing_flag',
    #'duacs_phase_screen',
    #'duacs_phase_screen_orbit',
    #'duacs_phase_screen_static',
    'duacs_relative_vorticity',
    'duacs_speed_meridional',
    'duacs_speed_meridional_abs',
    'duacs_speed_zonal',
    'duacs_speed_zonal_abs',
    'duacs_strain',
    'duacs_xcal',
    'sig0_karin_2',
    'phi',
    'swh_model', 
    'ssh_karin_uncert',
    'pass_number',
]

# For swot 2km
from swot import interp_dss, rotate
def interp_coloc_one_cycle(dfr_, dsalti):#, vars_to_rotate=[]):
    cycle = dfr_.cycle_number.mean()
    dsalti_ = dsalti.sel(cycle_number = cycle)
    
    # ATTENTION : NEED TO REMOVE VARIABLES FOR WHICH LONGITUDE, LATITUDE ARE NOT COORDS
    dropv = []
    if 'pass_number' in dsalti_.keys() : dropv +=['pass_number']
    if 'cycle_number' in dsalti_.keys() : dropv +=['cycle_number']
    if 'npts' in dsalti_.keys() : dropv +=['npts']
    if 'cutoff' in dsalti_.keys() : dropv +=['cutoff']
    
    df_interp = interp_dss(dsalti_.drop_vars(dropv), dfr_.longitude.values, dfr_.latitude.values)
    df_out = pd.concat([dfr_.reset_index()[['row_number', 'longitude']].set_index('longitude'), df_interp.set_index('longitude')], axis=1).reset_index().set_index('row_number')
    #for v in vars_to_rotate :
    #    df_out[v[O]], df_out[v[1]] = rotate(df_out[v[O]], df_out[v[1]], df_out(phi))
    return df_out
    
def coloc_swot2km_cycle(dfr, dsalti):

    meta = interp_coloc_one_cycle(dfr[dfr.cycle_number==500].compute(), dsalti)
    
    df_out = dfr.groupby('cycle_number').apply(interp_coloc_one_cycle, dsalti, meta=meta)
    #DF = []
    #for cycle in dfr[dfr.pass_number==swath].cycle_number.unique():
    #    try : 
    #        dfr_ = dfr.where((dfr.pass_number==swath)&(dfr.cycle_number==cycle)).dropna()
    #        dsalti_ = dsalti.sel(cycle_number=cycle)
    #        DF.append(interp_coloc_one_cycle(dfr_, dsalti_, cycle))
    #    except : 
    #        print('no', cycle)
    #        continue
        #print(cycle)
    #df_out = pd.concat(DF)
    if 'npts' in dsalti.keys(): df_out['npts'] = dsalti.npts.values
    if 'cutoff' in dsalti.keys(): df_out['cutoff'] = dsalti.cutoff.values
    return df_out

# For L4
def coloc_L4(dfr, dsalti):
    lon = df.longitude.values
    lat = df.latitude.values
    t = pd.to_datetime(df.datetime).values
    ds0 = dsalti.interp(longitude=('z', lon), latitude=('z',lat), time= ('z',t))
    df0 = ds0.to_dataframe()
    df0.index.names = ['row_number'] 
    return df0

    

In [12]:
ddf3 = ddf.where(ddf.pass_number==3).dropna().persist()
dsalti3 = xr.open_dataset(alti_files[0].replace('*', '3'))
dfc = coloc_swot2km_cycle(ddf3, dsalti3).reset_index().set_index('row_number').sort_index().compute()


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/dataframe/core.py:5400: UserWarning: New index has same name as existing, this is a no-op.
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 370.56 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


longitude   latitude  \
cycle_number row_number                         
478          0            4.871798  40.779977   
             1            4.870605  40.780433   
             2            4.869407  40.780881   
             3            4.868205  40.781321   
             4            4.866998  40.781753   
...                            ...        ...   
578          4743364      4.586696  42.201519   
             4743365      4.585575  42.201732   
             4743366      4.584508  42.201993   
             4743367      4.583500  42.202299   
             4743368      4.582554  42.202649   

                         ancillary_surface_classification_flag  \
cycle_number row_number                                          
478          0                                             0.0   
             1                                             0.0   
             2                                             0.0   
             3                                             0.0   
             4                                             0.0   
...                                                        ...   
578          4743364                                       0.0   
             4743365                                       0.0   
             4743366                                       0.0   
             4743367                                       0.0   
             4743368                                       0.0   

                         cross_track_distance  distance_to_coast  \
cycle_number row_number                                            
478          0                   20103.045274       99033.742399   
             1                   20096.136811       99047.162439   
             2                   20080.600164       99072.212802   
             3                   20050.892228       99102.058192   
             4                   19848.125536       99132.976955   
...                                       ...                ...   
578          4743364            -38714.617274      105021.151794   
             4743365            -38727.425312      105037.201245   
             4743366            -38751.859998      105052.223935   
             4743367            -38789.536457      105061.765132   
             4743368            -38987.212183      104881.020878   

                         duacs_editing_flag  duacs_phase_screen  \
cycle_number row_number                                           
478          0                          0.0            0.000100   
             1                          0.0            0.000100   
             2                          0.0            0.000101   
             3                          0.0            0.000103   
             4                          0.0            0.000113   
...                                     ...                 ...   
578          4743364                    0.0            0.002698   
             4743365                    0.0            0.002695   
             4743366                    0.0            0.002689   
             4743367                    0.0            0.002682   
             4743368                    0.0            0.002658   

                         duacs_phase_screen_orbit  duacs_phase_screen_static  \
cycle_number row_number                                                        
478          0                           0.000198                   0.002289   
             1                           0.000196                   0.002282   
             2                           0.000194                   0.002269   
             3                           0.000189                   0.002248   
             4                           0.000175                   0.002185   
...                                           ...                        ...   
578          4743364                    -0.000005                  -0.001501   
             4743365                    -0.000010  

___________
# Find all L4 files

In [36]:
alti_files = glob(os.path.join(zarr_dir, 'before_coloc', 'L4_sealevel', '*'))
alti_files

['/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/L4_sealevel/L4_noswot_regional.nc',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/L4_sealevel/L4_withnadirswot.nc',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/L4_sealevel/L4_noswot_global.nc']

In [27]:
for f in alti_files : 
    dsalti = xr.open_dataset(f)
    dfout = coloc_L4(df, dsalti)
    path = os.path.join(zarr_dir, "coloc_files",'alti', 'alticoloc_'+f.split('/')[-1].replace('.nc', '_'+colocs_source))
    dfout.to_csv(path)
    print(path)

/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_L4_noswot_regional_10d_spectral_decomp_all_med_variational_10min_v0.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_L4_withnadirswot_10d_spectral_decomp_all_med_variational_10min_v0.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_L4_noswot_global_10d_spectral_decomp_all_med_variational_10min_v0.csv


___________
# Find all preprocessed swot-2km files

In [16]:
alti_files = glob(os.path.join(zarr_dir, 'before_coloc', 'preprocessed_swot', 'swot2km', '*', '*'))
alti_files = [f.replace('pass3', 'pass*') for f in alti_files if 'pass3' in f]
alti_files = [f for f in alti_files if ('general' in f) | ('diff_only' in f)]
alti_files

['/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot2km/general/pass*_swot2km_general.nc',
 '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot2km/diff_only/pass*_swot2km_diff_only.nc']

# Interpolate with the data of the nearest cycle


In [20]:
for i in range(len(alti_files)) :

    path = alti_files[i].replace('/'.join(alti_files[i].split('/')[6:10]), 'coloc_files/alti').replace('pass*', 'alticoloc').replace('.nc', '_'+colocs_source)
    #if os.path.isfile(path) :
    #   print(f'file already exists : {path}')
    #   continue
        
    # Alti files
    files = glob(alti_files[i])
    D=[]# for over pass_number
    for f in files:
        try : 
            dsalti = xr.open_dataset(f).compute()
            if 'phi' in dsalti.variables: dsalti = dsalti.reset_coords(['phi', 'dx', 'dy'])
            pass_number = dsalti.pass_number.values
            print(pass_number)
            ddf_ = ddf[ddf.pass_number==pass_number].persist()
            df_out = coloc_swot2km_cycle(ddf_, dsalti).compute().reset_index().set_index('row_number').sort_index()
            df_out['pass_number']=int(pass_number)
            D.append(df_out)
        except :
            assert False, f'pb with {f}'
    pd.concat(D, axis=0).to_csv(path)
    print(path)
    

16


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 347.92 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


3


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 370.56 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_swot2km_general_10d_spectral_decomp_all_med_variational_10min_v0.csv
3


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 295.42 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


16


/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 277.36 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_swot2km_diff_only_10d_spectral_decomp_all_med_variational_10min_v0.csv


______________________
# Find L3-250m

In [6]:
alti_dir = glob(os.path.join(zarr_dir, 'before_coloc', 'preprocessed_swot', 'swot250m', '*', '*'))
#alti_files = [f.replace('pass3', 'pass*') for f in alti_files if 'pass3' in f]
alti_dir = np.unique(['/'.join(f.split('/')[:-1]) for f in alti_dir])
alti_dir

array(['/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/diff_only',
       '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/gaussian_10000',
       '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/gaussian_15000',
       '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/gaussian_20000',
       '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/gaussian_25000',
       '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/gaussian_30000',
       '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/gaussian_35000',
       '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/gaussian_40000',
       '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_co

In [16]:
def wrapper_interp_one_cycle(df, dir_):
    #print(df.keys)
    swath = int(df.reset_index().pass_number.mean())
    cycle = int(df.reset_index().cycle_number.mean())
    path = os.path.join(dir_, f'{int(swath)}_{int(cycle)}.nc')
    try : 
        dsalti = xr.open_dataset(path)
        if 'phi' in dsalti : dsalti = dsalti.reset_coords(['phi'])
        print(path)
    except : 
        assert False, path
    dfout = interp_coloc_one_cycle(df, dsalti, int(cycle))
    return dfout
    
dir_ = alti_dir[-1]
dfr = df.where((df.pass_number==3)&(df.cycle_number==500)).dropna().iloc[0:100]
meta = dfr.groupby(['pass_number', 'cycle_number']).apply(wrapper_interp_one_cycle, dir_)
meta

/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/general/3_500.nc


/dev/shm/pbs.2672310.datarmor0/ipykernel_21019/3523304540.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  meta = dfr.groupby(['pass_number', 'cycle_number']).apply(wrapper_interp_one_cycle, dir_)


longitude   latitude  \
pass_number cycle_number row_number                         
3.0         500.0        53513        5.045020  40.286939   
                         53514        5.045803  40.286533   
                         53515        5.046585  40.286116   
                         53516        5.047365  40.285685   
                         53517        5.048142  40.285242   
...                                        ...        ...   
                         53608        5.113086  40.229164   
                         53609        5.113920  40.228469   
                         53610        5.114743  40.227760   
                         53611        5.115556  40.227040   
                         53612        5.116359  40.226308   

                                     ancillary_surface_classification_flag  \
pass_number cycle_number row_number                                          
3.0         500.0        53513                                         0.0   
                         53514                                         0.0   
                         53515                                         0.0   
                         53516                                         0.0   
                         53517                                         0.0   
...                                                                    ...   
                         53608                                         0.0   
                         53609                                         0.0   
                         53610                                         0.0   
                         53611                                         0.0   
                         53612                                         0.0   

                                     cross_track_distance  distance_to_coast  \
pass_number cycle_number row_number                                            
3.0         500.0        53513               47884.371205       73436.919437   
                         53514               47964.445342       73748.316242   
                         53515               48022.723029       73745.809814   
                         53516               48096.306429       73726.202613   
                         53517               48197.370788       73848.036583   
...                                                   ...                ...   
                         53608               55024.535314       76163.125365   
                         53609               55092.468060       76369.872240   
                         53610               55224.584934       76850.455710   
                         53611               55269.123733       76819.005490   
                         53612               55358.220072       76660.926778   

                                     duacs_editing_flag  duacs_phase_screen  \
pass_number cycle_number row_number                                           
3.0         500.0        53513                      0.0            0.001393   
                         53514                      0.0            0.001328   
                         53515                      0.0            0.001297   
                         53516                      0.0            0.001261   
                         53517                      0.0            0.001216   
...                                                 ...                 ...   
                         53608                      0.0           -0.003720   
                         53609                      0.0           -0.003774   
                         53610                      0.0           -0.003880   
                         53611                      0.0           -0.003928   
                         53612                      0.0           -0.004030   

                                     duacs_phase_screen_orbit  \
pass_number cycle_number row_number                             
3.0         500.0        5351

In [18]:
meta.columns

Index(['longitude', 'latitude', 'ancillary_surface_classification_flag',
       'cross_track_distance', 'distance_to_coast', 'duacs_editing_flag',
       'duacs_phase_screen', 'duacs_phase_screen_orbit',
       'duacs_phase_screen_static', 'duacs_relative_vorticity',
       'duacs_speed_meridional', 'duacs_speed_meridional_abs',
       'duacs_speed_zonal', 'duacs_speed_zonal_abs', 'duacs_strain',
       'duacs_xcal', 'sig0_karin_2', 'phi', 'swh_model'],
      dtype='object')

In [19]:
for dir_ in alti_dir[-2:] :
    path = dir_.replace('before_coloc/preprocessed_swot', 'coloc_files/alti').replace('swot250m/', 'alticoloc_swot250m_')+'_'+colocs_source
    #if os.path.isfile(path) : continue
    
    dfr = df.where((df.pass_number==3)&(df.cycle_number==500)).dropna().iloc[0:100]
    meta = dfr.groupby(['pass_number', 'cycle_number']).apply(wrapper_interp_one_cycle, dir_)
    
    df_out = ddf.groupby(['pass_number', 'cycle_number']).apply(wrapper_interp_one_cycle, dir_,  meta = meta).reset_index().compute()
    path = dir_.replace('before_coloc/preprocessed_swot', 'coloc_files/alti').replace('swot250m/', 'alticoloc_swot250m_')+'_'+colocs_source
    df_out.to_csv(path)
    print(path)
    

/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/gaussian_55000/3_500.nc


/dev/shm/pbs.2672310.datarmor0/ipykernel_21019/3135878760.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  meta = dfr.groupby(['pass_number', 'cycle_number']).apply(wrapper_interp_one_cycle, dir_)


/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_swot250m_gaussian_55000_12h_spectral_decomp_all_med_variational_10min_v0.csv
/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/before_coloc/preprocessed_swot/swot250m/general/3_500.nc


/dev/shm/pbs.2672310.datarmor0/ipykernel_21019/3135878760.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  meta = dfr.groupby(['pass_number', 'cycle_number']).apply(wrapper_interp_one_cycle, dir_)


/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_swot250m_general_12h_spectral_decomp_all_med_variational_10min_v0.csv


In [20]:
path = '/home/datawork-lops-oc/aponte/margot/DATA_MED_COLOC/coloc_files/alti/alticoloc_swot250m_general_12h_spectral_decomp_all_med_variational_10min_v0.csv'
df = pd.read_csv(path)
df.columns

Index(['Unnamed: 0', 'pass_number', 'cycle_number', 'row_number', 'longitude',
       'latitude', 'ancillary_surface_classification_flag',
       'cross_track_distance', 'distance_to_coast', 'duacs_editing_flag',
       'duacs_phase_screen', 'duacs_phase_screen_orbit',
       'duacs_phase_screen_static', 'duacs_relative_vorticity',
       'duacs_speed_meridional', 'duacs_speed_meridional_abs',
       'duacs_speed_zonal', 'duacs_speed_zonal_abs', 'duacs_strain',
       'duacs_xcal', 'sig0_karin_2', 'phi', 'swh_model'],
      dtype='object')